In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, count, desc, round as _round


In [2]:
spark = (
    SparkSession.builder
    .appName("SIMBA_Analytics")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .config("spark.driver.extraJavaOptions", "-Dhadoop.home.dir=E:/hadoop")
    .master("local[*]")
    .getOrCreate()
)

JDBC_URL = "jdbc:postgresql://localhost:5432/simba_db"
DB_PROPS = {
    "user": "postgres",
    "password": "12344321",
    "driver": "org.postgresql.Driver"
}


In [3]:
# Load data ke DataFrame Spark
df = spark.read.jdbc(url=JDBC_URL, table="clean_sales_transactions", properties=DB_PROPS)

# Cache data agar proses berikutnya lebih cepat (data disimpan di RAM)
df.cache()

print(f"Berhasil menarik {df.count():,} baris data.")
df.show(5)


Berhasil menarik 522,602 baris data.
+--------------+----------------+----+-----+---+-----------+----------+--------------------+----------+--------+----------+-----------+--------------+
|transaction_no|transaction_date|year|month|day|day_of_week|product_no|        product_name|unit_price|quantity|line_total|customer_no|       country|
+--------------+----------------+----+-----+---+-----------+----------+--------------------+----------+--------+----------+-----------+--------------+
|        581482|      2019-12-09|2019|   12|  9|     Monday|     22485|Set Of 2 Wooden M...|     21.47|      12|    257.64|      17490|United Kingdom|
|        581475|      2019-12-09|2019|   12|  9|     Monday|     22596|Christmas Star Wi...|     10.65|      36|    383.40|      13069|United Kingdom|
|        581475|      2019-12-09|2019|   12|  9|     Monday|     23235|Storage Tin Vinta...|     11.53|      12|    138.36|      13069|United Kingdom|
|        581475|      2019-12-09|2019|   12|  9|     Mond

In [4]:
monthly_processing = (
    df.groupBy("year", "month")
    .agg(
        _sum("quantity").alias("total_units_sold"),
        _round(_sum("line_total"), 2).alias("total_revenue")
    )
    .orderBy("year", "month")
)

print("=== HASIL PROCESSING: AGREGASI BULANAN ===")
monthly_processing.show()


=== HASIL PROCESSING: AGREGASI BULANAN ===
+----+-----+----------------+-------------+
|year|month|total_units_sold|total_revenue|
+----+-----+----------------+-------------+
|2018|   12|          357407|   4397648.39|
|2019|    1|          386332|   4548423.47|
|2019|    2|          282587|   3327342.64|
|2019|    3|          376809|   4384669.82|
|2019|    4|          307509|   3579310.06|
|2019|    5|          394572|   4569952.21|
|2019|    6|          388543|   4486132.15|
|2019|    7|          398750|   4571494.88|
|2019|    8|          421286|   4749801.23|
|2019|    9|          568844|   6613772.79|
|2019|   10|          619284|   7212279.85|
|2019|   11|          760175|   7828489.53|
|2019|   12|          312752|   2512069.52|
+----+-----+----------------+-------------+



In [5]:
# Analisis Produk Terlaris
top_products = (
    df.groupBy("product_no", "product_name")
    .agg(_sum("quantity").alias("units_sold"))
    .orderBy(desc("units_sold"))
    .limit(10)
)

# Analisis Penjualan per Negara
country_analytics = (
    df.groupBy("country")
    .agg(_round(_sum("line_total"), 2).alias("revenue"))
    .orderBy(desc("revenue"))
)

print("=== TOP 10 PRODUK ===")
top_products.show(truncate=False)

print("=== REVENUE PER NEGARA ===")
country_analytics.show(5)


=== TOP 10 PRODUK ===
+----------+----------------------------------+----------+
|product_no|product_name                      |units_sold|
+----------+----------------------------------+----------+
|23843     |Paper Craft Little Birdie         |80995     |
|23166     |Medium Ceramic Top Storage Jar    |78033     |
|22197     |Popcorn Holder                    |56902     |
|84077     |World War 2 Gliders Asstd Designs |54951     |
|85099B    |Jumbo Bag Red Retrospot           |48375     |
|85123A    |Cream Hanging Heart T-Light Holder|37937     |
|21212     |Pack Of 72 Retrospot Cake Cases   |36492     |
|84879     |Assorted Colour Bird Ornament     |36394     |
|23084     |Rabbit Night Light                |30742     |
|22492     |Mini Paint Set Vintage            |26633     |
+----------+----------------------------------+----------+

=== REVENUE PER NEGARA ===
+--------------+-----------+
|       country|    revenue|
+--------------+-----------+
|United Kingdom|52346877.60|
|   Neth

In [6]:
# Simpan hasil olahan ke tabel baru di database
monthly_processing.write.jdbc(
    url=JDBC_URL,
    table="agg_monthly_sales",
    mode="overwrite",
    properties=DB_PROPS
)

print("Hasil Processing & Analytics berhasil disimpan ke tabel 'agg_monthly_sales' di PostgreSQL.")


Hasil Processing & Analytics berhasil disimpan ke tabel 'agg_monthly_sales' di PostgreSQL.
